In [ ]:
import swissparlpy as spp
from swissparlpy.backends import ODataBackend, OpenParlDataBackend
import logging
import pandas as pd
from typing import cast

from pprint import pprint

In [ ]:
loglevel = logging.DEBUG
logging.basicConfig(
    format="%(asctime)s %(name)s %(levelname)-8s %(message)s",
    level=loglevel,
    datefmt="%Y-%m-%d %H:%M:%S",
)
logging.captureWarnings(True)

## Tables from OData backend:

In [ ]:
# Get tables from both backends
odata_tables = spp.get_tables(backend="odata")
print(f"  Count: {len(odata_tables)}")
print(f"  Sample: {odata_tables[:3]}")

Tables from OpenParlData backend:

In [ ]:
opd_tables = spp.get_tables(backend="openparldata")
print(f"  Count: {len(opd_tables)}")
print(f"  Sample: {opd_tables[:3]}")

## Using SwissParlClient directly with specific backends

In [ ]:
odata_backend = ODataBackend()
odata_client = spp.SwissParlClient(backend=odata_backend)
print(f"\nOData client tables count: {len(odata_client.get_tables())}")

In [ ]:
opd_backend = OpenParlDataBackend()
opd_client = spp.SwissParlClient(backend=opd_backend)
print(f"OpenParlData client tables count: {len(opd_client.get_tables())}")

Print all tables with their properties


In [ ]:
# OData client (parlament.ch API) overview

overview = odata_client.get_overview()
for table, props in overview.items():
    print(table)
    for prop in props:
        print(f" + {prop}")
    print("")

In [ ]:
# OpenParlData client (openparldata.ch API) overview

overview = opd_client.get_overview()
for table, props in overview.items():
    print(table)
    for prop in props:
        print(f" + {prop}")
    print("")

In [ ]:
opd_client.get_variables("speeches")

In [ ]:
opd_client.get_variables("votes")

In [ ]:
response = opd_client.get_data("votes")
response

In [ ]:
print(f"Loaded: {response._records_loaded_count}")
print(f"Total records: {len(response)}")

In [ ]:
type(response[501])

In [ ]:
pprint(response[1001])
print(f"Loaded: {response._records_loaded_count}")

## Load data of a person

In [ ]:
response = opd_client.get_data(
    "persons", firstname="Maya", lastname="Graf", body_key="CHE"
)
person = cast(spp.OpenParlDataResponse, response[0])
pprint(person)

In [ ]:
response.to_dataframe()

Get related data of an entry (here "memberships" of a person)

In [ ]:
memberships = person.get_related_data("memberships")
pprint(memberships)

In [ ]:
print(f"len: {len(memberships)}")
print(f"type: {type(memberships)}")
print(f"type of first element: {type(memberships[0])}")

In [ ]:
pprint(memberships[0])

In [ ]:
pd.DataFrame(memberships[0:99])

Since the API currently gives wrong URLs for the pagination ("person" instead of "persons"), this call currently fails:

In [ ]:
memberships.to_dataframe()

In [ ]:
response = opd_client.get_data(
    "persons", firstname="Maya", lastname="Graf", body_key="CHE", expand="votes"
)
response[0]["votes"]

In [ ]:
response[0]

Get memberships of a person

In [ ]:
memres = opd_client.get_data(
    "memberships", person_id=9527, type_harmonized="Arbeitsgruppe"
)
memres.to_dataframe().iloc[0]["links"]

Get the related group for a membership:

In [ ]:
groups = memres[0].get_related_data("group")
print(f"len: {len(groups)}")
print(f"type: {type(groups)}")
print(f"type of first element: {type(groups[0])}")

In [ ]:
opd_client = spp.SwissParlClient(backend="openparldata")
opd_client.get_tables()[0:5]

In [ ]:
response = opd_client.get_data("persons", firstname="Karin", lastname="Keller-Sutter")
response.to_dataframe()

In [ ]:
response = opd_client.get_data("speeches", search_mode="natural", search_scope="all", search_language="de", search="Budget")
len(response)

In [ ]:
speech_df = response.to_dataframe()
speech_df[["id", "body_key", "person_id", "meeting_id", "date_start", "date_end", "text_content_de"]]

In [ ]:
speech_df.columns

In [ ]:
speech = cast(spp.OpenParlDataResponse, response[0])
speech.get_related_data("bodies")

In [ ]:
data = spp.get_data('persons', firstname="Stefan", backend="openparldata")
data.count